<div style="background-color: #ADD8E6; border: 1px solid gray; padding: 3px">
    <h3>Code Spec Evaluation</h3>
    The following is an overview of the workflow:
    <ul>
    <li>Uses <b>DeepEval</b> to evaluate the code specifications doc</li>
    <li>Uses specification generated by gpt-oss-20b as reference set</li>
    </ul>
</div>

In [1]:
##############################################
# Imports
##############################################
import os
import traceback
from deepeval.models.base_model import DeepEvalBaseLLM
from deepeval import assert_test
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.metrics import GEval, AnswerRelevancyMetric
from deepeval import evaluate
from evaluate import load
from deepeval.metrics import ArenaGEval
from deepeval.test_case import ArenaTestCase
from deepeval import compare
import utils
import nltk
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
load_dotenv()
import nest_asyncio
nest_asyncio.apply()

In [ ]:
##############################################
# Set Up Metric Instances
##############################################
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('punkt_tab')
bleu_metric = load("bleu")
rouge_metric = load("rouge")
meteor_metric = load("meteor")
bertscore_metric = load("bertscore")

In [ ]:
##############################################
# Evaluation Methods
##############################################
def compute_eval_scores(predictions, references):
    def compute_bleu4_scores(predictions, references):
        """
        Computes BLEU-4 Scores
        """
        results = bleu_metric.compute(predictions=predictions, references=references)
        return results["bleu"]
    
    def compute_rouge_scores(predictions, references):
        """
        Computes ROUGE Scores
        """
        results = rouge_metric.compute(predictions=predictions, references=references)
        return results["rougeL"]
    
    def compute_meteor_scores(predictions, references):
        """
        Computes METEOR Scores
        """
        results = meteor_metric.compute(predictions=predictions, references=references)
        return results["meteor"]
        
    
    def compute_bert_scores(predictions, references):
        """
        Computes BERT Scores
        """
        results = bertscore_metric.compute(predictions=predictions, references=references, lang="en")
        return results["f1"]

    return {
        "bleu4": compute_bleu4_scores(predictions, references),
        "rougel": compute_rouge_scores(predictions, references),
        "meteor": compute_meteor_scores(predictions, references),
        "bert": compute_bert_scores(predictions, references)
    }

In [ ]:
##############################################
# Utilities
##############################################
def get_file_content(filepath: str) -> str:
    with open(filepath, "r") as file:
        return file.read()

### Run Evaluations
Run the evaluations pipeline!

In [ ]:
def evaluations_pipeline(content: str, reference_content: str):

    data = compute_eval_scores([content], [reference])

    return pd.DataFrame(data)
    

In [ ]:
app_name = "golfap"

bucket_name = "data"

candidate_content = utils.fetch_indexing_job(app_name, bucket_name)

reference_content = get_file_content("spec/reference_spec.md")

candidate_results = evaluations_pipeline(candidate_content, reference_content)

candidate_results

In [ ]:
app_name = "golfap"

bucket_name = "data"

baseline_content = get_file_content("spec/baseline_spec.md")

baseline_results = evaluations_pipeline(baseline_content, reference_content)

baseline_results